# Step by step explanation of Adaboost

## Dataset

| X1 | X2 | Y |
|----|----|---|
| 3  | 7  | 1 |
| 2  | 9  | 0 |
| 1  | 4  | 1 |
| 9  | 8  | 0 |
| 3  | 7  | 0 |

# AdaBoost: Step-by-Step (Stage 1)

## Initial Setup

- Number of data points: $$n = 5$$  
- Initial weight for each data point:

$$
w_i = \frac{1}{n} = \frac{1}{5} = 0.2
$$

A new column of weights is created where each row has weight $$0.2$$.

---

## Stage 1: Train First Decision Stump

- Train a decision tree with:

$$
\text{max\_depth} = 1
$$

- Choose the stump that gives:
  - Maximum information gain  
  - Maximum reduction in entropy  

Let this model be:

$$
m_1
$$

---

## Stage 2: Predictions and Error

- Create a new column:

$$
y_{\text{pred}}
$$

- Compare predictions with actual values

- Compute error:

$$
\text{error} = \sum (\text{weights of misclassified points})
$$

Example:
- If 2 points are misclassified:

$$
\text{error} = 0.2 + 0.2 = 0.4
$$

Note:
- Error is **not** equal to $$1 - \text{accuracy}$$  
- It depends on **weights**, not just count  

---

## Model Reliability Intuition

Consider three models:

- $$m_1$$ → error = 0%  
- $$m_2$$ → error = 100%  
- $$m_3$$ → error = 50%  

Insights:
- overall Best model among all three is $$m_1$$   
- if we remove m1 what is the best model in m2 and m3: 
  - $$m_2$$ can be flipped to become perfect  
  - $$m_3$$ is unreliable (random behavior)

---

## Alpha Calculation

Alpha represents model importance.

$$
\alpha = \frac{1}{2} \ln \left( \frac{1 - \text{error}}{\text{error}} \right)
$$

Example:

$$
\text{error} = 0.4
$$

$$
\alpha_1 = \frac{1}{2} \ln \left( \frac{1 - 0.4}{0.4} \right) \approx 0.20
$$

- Lower error → higher $$\alpha$$  
- Higher error → lower or negative $$\alpha$$  

---

## Weight Update (Boosting Step)

Now we update weights to focus on mistakes.

### For Misclassified Points:

$$
w_i^{\text{new}} = w_i \cdot e^{\alpha_1}
$$

### For Correctly Classified Points:

$$
w_i^{\text{new}} = w_i \cdot e^{-\alpha_1}
$$

- Misclassified → weight increases  
- Correct → weight decreases  

---

## Updated Weights Column

After applying above formulas:
- Create a new column:

$$
w_i^{\text{updated}}
$$

---

## Normalization

After updating, weights may not sum to 1.

So we normalize:

$$
w_i^{\text{normalized}} = \frac{w_i^{\text{updated}}}{\sum w_i^{\text{updated}}}
$$

Now:

$$
\sum w_i^{\text{normalized}} = 1
$$

---

## Summary of Stage 1

- Initialize equal weights  
- Train first stump $$m_1$$  
- Compute weighted error  
- Calculate $$\alpha_1$$  
- Update weights using exponential rule  
- Normalize weights  

Now the model is ready for the next stump, which will focus more on previously misclassified points.

---------

## Stage 1 Table

| X1 | X2 | Y | Y_pred | Weight | Updated Weight | Normalized Weight |
|----|----|---|--------|--------|----------------|-------------------|
| 3  | 7  | 1 | 1      | 0.2    | 0.16           | 0.166             |
| 2  | 9  | 0 | 1      | 0.2    | 0.24           | 0.25              |
| 1  | 4  | 1 | 0      | 0.2    | 0.24           | 0.25              |
| 9  | 8  | 0 | 0      | 0.2    | 0.16           | 0.166             |
| 3  | 7  | 0 | 0      | 0.2    | 0.16           | 0.166             |

### Notes

- Misclassified points:
  - Row 2 and Row 3  
- Error:

$$
\text{error} = 0.2 + 0.2 = 0.4
$$

- Alpha:

$$
\alpha_1 = 0.20
$$

- Sum of updated weights:

$$
0.16 + 0.24 + 0.24 + 0.16 + 0.16 = 0.96
$$

- Normalization:

$$
w_i^{\text{normalized}} = \frac{w_i^{\text{updated}}}{0.96}
$$

------------

## Sampling Table (After Normalization)

| X1 | X2 | Y | New Weight | Range        |
|----|----|---|------------|--------------|
| 3  | 7  | 1 | 0.166      | 0 – 0.166    |
| 2  | 9  | 0 | 0.25       | 0.166 – 0.416|
| 1  | 4  | 1 | 0.25       | 0.416 – 0.666|
| 9  | 8  | 0 | 0.166      | 0.666 – 0.832|
| 3  | 7  | 0 | 0.166      | 0.832 – 1.0  |

## Upsampling (Weighted Sampling)

After computing normalized weights, we perform **upsampling** to create a new dataset for the next decision stump.

---

### Step 1: Generate Random Numbers

- Generate $$n$$ random numbers between $$0$$ and $$1$$  
- Here $$n = 5$$  

Example:

$$
0.13,\; 0.43,\; 0.62,\; 0.50,\; 0.80
$$

---

### Step 2: Select Rows Based on Range

- Each data point has a range (from previous table)  
- For each random number:
  - Check which range it falls into  
  - Select that corresponding row  

---

### Example Mapping

| Random Number | Selected Row |
|--------------|-------------|
| 0.13         | 1           |
| 0.43         | 3           |
| 0.62         | 3           |
| 0.50         | 3           |
| 0.80         | 4           |

Final selected rows:

$$
[1, 3, 3, 3, 4]
$$

---

### Key Insight

- Rows with **higher weight** have **larger ranges**  
- Larger range → higher probability of selection  
- That is why **Row 3 appears multiple times**

---

### Step 3: Train Next Decision Stump

- Use only the sampled dataset:
  
$$
[1, 3, 3, 3, 4]
$$

- Train new decision stump:

$$
m_2
$$

- Compute:
  - Error  
  - $$\alpha_2$$  

---

### Step 4: Repeat the Process

- Update weights again  
- Perform upsampling again  
- Train next stump  

Repeat this process for $$n$$ decision stumps:

$$
\alpha_1, \alpha_2, \alpha_3, \dots, \alpha_n
$$

---

## Final Prediction

All models are combined using weighted voting:

$$
h(x) = \text{sign} \big( \alpha_1 h_1(x) + \alpha_2 h_2(x) + \alpha_3 h_3(x) \big)
$$

---

## Summary

- Upsampling increases focus on difficult points  
- High-weight points are selected multiple times  
- Each new model learns from previous mistakes  
- Final prediction is based on weighted combination of all models  